# B1.10 · Exploit chaining

**Function B — Product & Application Security → The AppSec Engineer / Code Reviewer**  ·  *AI for Security*

Builds on **[B1.9 · Dynamic exploitation (DAST)](https://spbreed.github.io/cyber-commons/lessons/B1.9.html)**.

| | |
|---|---|
| Open-source tooling | OWASP ZAP |
| Open-weight models | Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


**Stage 13 — Exploit chaining.** Individual findings are triaged individually,
and that is how three mediums become a critical nobody noticed.

The arithmetic of severity is not additive. A read-only information disclosure
is a medium. A CSRF is a medium. An unauthenticated internal endpoint is a
medium. Chained — leak an ID, forge a request using it, hit the internal
endpoint with the forged session — the outcome is account takeover, which is
not a medium.

The pipeline can find these mechanically because Phase 4 already produced
confirmed findings with known **preconditions** and **effects**. If one
finding's effect satisfies another's precondition, they compose, and the chain's
severity is the severity of its final effect.

This is the stage that most often changes what gets fixed first.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 2 · Stage 13 — findings as preconditions and effects

In [ ]:
from dataclasses import dataclass, field
from itertools import permutations

@dataclass(frozen=True)
class Confirmed:
    fid: str; cwe: str; name: str
    requires: frozenset       # preconditions
    grants: frozenset         # effects
    severity: str

FINDINGS = [
 Confirmed("F-01","CWE-200","report id disclosed in an error message",
           frozenset({"unauthenticated"}), frozenset({"valid_report_id"}), "low"),
 Confirmed("F-02","CWE-639","report fetch does not check ownership (IDOR)",
           frozenset({"valid_report_id","authenticated"}),
           frozenset({"other_users_report_data"}), "medium"),
 Confirmed("F-03","CWE-352","password change endpoint lacks CSRF protection",
           frozenset({"authenticated"}), frozenset({"password_reset_for_victim"}), "medium"),
 Confirmed("F-04","CWE-89","SQL injection in the report filter",
           frozenset({"authenticated"}), frozenset({"database_read"}), "high"),
 Confirmed("F-05","CWE-306","internal admin endpoint has no authentication",
           frozenset({"internal_network"}), frozenset({"admin_actions"}), "medium"),
 Confirmed("F-06","CWE-918","report export fetches a user-supplied URL (SSRF)",
           frozenset({"authenticated"}), frozenset({"internal_network"}), "medium"),
]
SEV_RANK = {"low":1,"medium":2,"high":3,"critical":4}
START = frozenset({"unauthenticated","authenticated"})

print(f"{'id':7s}{'cwe':10s}{'sev':9s}{'requires':38s}grants")
print("-" * 96)
for f in FINDINGS:
    print(f"{f.fid:7s}{f.cwe:10s}{f.severity:9s}{str(sorted(f.requires)):38s}"
          f"{sorted(f.grants)}")

## 3 · Compose them — an effect that satisfies the next precondition

In [ ]:
EFFECT_SEVERITY = {
 "other_users_report_data": "high",
 "password_reset_for_victim": "critical",
 "admin_actions": "critical",
 "database_read": "high",
 "internal_network": "medium",
 "valid_report_id": "low",
}

def chains(findings, start, max_len=4):
    found = []
    def walk(path, state):
        if len(path) >= max_len: return
        for f in findings:
            if f in path: continue
            if not f.requires <= state: continue
            new_state = state | f.grants
            new_path = path + [f]
            if len(new_path) > 1:
                worst = max((EFFECT_SEVERITY.get(g, "low") for g in f.grants),
                            key=lambda s: SEV_RANK[s])
                found.append({"chain": new_path, "final_effect": sorted(f.grants),
                              "severity": worst})
            walk(new_path, new_state)
    walk([], start)
    return found

ALL = chains(FINDINGS, START)
best = {}
for c in ALL:
    key = tuple(f.fid for f in c["chain"])
    best[key] = c
ranked = sorted(best.values(), key=lambda c: (-SEV_RANK[c["severity"]], len(c["chain"])))

print(f"{len(ranked)} composable chains found\n")
for c in ranked[:6]:
    ids = " → ".join(f.fid for f in c["chain"])
    links = ", ".join(f"{f.severity}" for f in c["chain"])
    print(f"[{c['severity']:8s}] {ids:26s} links: {links}")
    print(f"{'':11s} final effect: {c['final_effect']}")

## 4 · Where it breaks — triage the links, miss the chain

In [ ]:
individual = max(SEV_RANK[f.severity] for f in FINDINGS)
chained = max(SEV_RANK[c["severity"]] for c in ranked)
inv = {v: k for k, v in SEV_RANK.items()}
print(f"highest individual finding severity : {inv[individual]}")
print(f"highest chained severity            : {inv[chained]}")

critical_chains = [c for c in ranked if c["severity"] == "critical"]
print(f"\ncritical chains built entirely from non-critical findings:")
for c in critical_chains[:3]:
    ids = " → ".join(f"{f.fid}({f.severity})" for f in c["chain"])
    print(f"   {ids}")
    print(f"      → {c['final_effect']}")

all_links_medium_or_below = [c for c in critical_chains
                             if all(SEV_RANK[f.severity] <= 2 for f in c["chain"])]
print(f"\n{len(all_links_medium_or_below)} critical chain(s) whose every link is "
      f"medium or lower.")
print("Triaged individually, none of those findings would be worked this sprint.")
assert all_links_medium_or_below

In [ ]:
# The control: rank by chain severity, and report the chain, not the link.
def remediation_order(findings, chains_):
    """Fixing one link breaks every chain through it. Rank by chains broken."""
    impact = {}
    for f in findings:
        broken = [c for c in chains_ if f in c["chain"]]
        worst = max((SEV_RANK[c["severity"]] for c in broken), default=0)
        impact[f.fid] = {"chains_broken": len(broken), "worst_chain": inv.get(worst, "—"),
                         "own_severity": f.severity}
    return sorted(impact.items(),
                  key=lambda kv: (-SEV_RANK.get(kv[1]["worst_chain"], 0),
                                  -kv[1]["chains_broken"]))

print(f"{'finding':9s}{'own sev':10s}{'chains broken':>15}{'worst chain':>14}")
print("-" * 50)
for fid, i in remediation_order(FINDINGS, ranked):
    print(f"{fid:9s}{i['own_severity']:10s}{i['chains_broken']:>15}{i['worst_chain']:>14}")
top = remediation_order(FINDINGS, ranked)[0]
print(f"\nfix first: {top[0]} — own severity {top[1]['own_severity']}, but it "
      f"breaks {top[1]['chains_broken']} chains including a {top[1]['worst_chain']}")

## What you just proved

Six confirmed findings compose into multiple chains. The highest individual severity is high while the highest chained severity is critical, and at least one critical chain is built entirely from medium-or-lower links — for example SSRF granting internal network access, then the unauthenticated admin endpoint. Remediation ordering puts a medium finding first because it breaks the most chains.

## Your turn

Take your current open findings and write down each one's preconditions and effects. The chaining falls out mechanically, and the finding you should fix first is usually not the one at the top of the severity-sorted queue.

---

**Next → [B1.11 · Remediation engineering](https://spbreed.github.io/cyber-commons/lessons/B1.11.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.10.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.10.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*